In [46]:
import chardet

with open("LIS_data.csv", "rb") as f:
    raw = f.read()
    result = chardet.detect(raw)

print(result)

{'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': ''}


In [47]:
import pandas as pd

# 1. Load the CSV file using a relative path
file_path = "LIS_data.csv"
df = pd.read_csv(file_path)

# 2. Check basic information about the DataFrame
print("=== DataFrame Information ===")
print(df.info())

# 3. Preview the first few rows of the DataFrame
print("\n=== DataFrame Preview (head) ===")
print(df.head())


=== DataFrame Information ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80794 entries, 0 to 80793
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Author full names              80729 non-null  object
 1   Author(s) ID                   80730 non-null  object
 2   Title                          80794 non-null  object
 3   Year                           80794 non-null  int64 
 4   Source title                   80794 non-null  object
 5   Affiliations                   80067 non-null  object
 6   Funding Details                32444 non-null  object
 7   Funding Texts                  30014 non-null  object
 8   Language of Original Document  80794 non-null  object
 9   Document Type                  80794 non-null  object
dtypes: int64(1), object(9)
memory usage: 6.2+ MB
None

=== DataFrame Preview (head) ===
                                                                

In [48]:
# Step 1: Split funding organizations by semicolon
# ------------------------------------------------
# The "Funding Details" column may contain multiple funding organizations
# separated by semicolons (;). If there is only one organization, no split is needed.
# We will convert each cell into a list of funding organizations.

df["Funding_Details_Split"] = df["Funding Details"].apply(
    lambda x: [item.strip() for item in str(x).split(";")] if pd.notnull(x) else []
)

# Ensure full column content is displayed
pd.set_option('display.max_colwidth', None)

print(df[["Funding Details", "Funding_Details_Split"]].head(10))

                                                                                                                    Funding Details  \
0                                                                                                                               NaN   
1                                                                                                                               NaN   
2                                                             Hrvatska Zaklada za Znanost, HRZZ, (IP-2014-09-7672, IP-2019-04-4882)   
3                                                                                                                               NaN   
4                                                                                                                               NaN   
5                                                                                                                               NaN   
6                                                      

In [49]:
import re

# Step 2: Remove grant numbers inside parentheses
# ----------------------------------------------
# Some funding organization names include grant numbers in parentheses,
# such as "(62372247, 12101326)". We will remove any content inside parentheses,
# including the parentheses themselves.

def remove_parentheses(text):
    # Remove any text inside parentheses and strip extra whitespace
    return re.sub(r"\s*\(.*?\)", "", text).strip()

# Apply the function to each funding organization in the list
df["Funding_Details_Clean"] = df["Funding_Details_Split"].apply(
    lambda lst: [remove_parentheses(item) for item in lst]
)

# Optional: Check the result
print(df[["Funding_Details_Split", "Funding_Details_Clean"]].head(10))

                                                                                                                Funding_Details_Split  \
0                                                                                                                                  []   
1                                                                                                                                  []   
2                                                             [Hrvatska Zaklada za Znanost, HRZZ, (IP-2014-09-7672, IP-2019-04-4882)]   
3                                                                                                                                  []   
4                                                                                                                                  []   
5                                                                                                                                  []   
6                                        

In [50]:
import re

# Step 3 (advanced): Remove only trailing abbreviations or alternate-language aliases
# -----------------------------------------------------------------------------------
# This version is more conservative than simply removing everything after the last comma.
# It keeps legitimate institution names such as:
# "University of California, Berkeley"
# but removes trailing abbreviations or alias-like endings such as:
# "National Natural Science Foundation of China, NSFC,"
# "Research Grants Council, University Grants Committee, 研究資助局,"
#
# Heuristic rules:
# 1. Remove the final segment if it looks like an abbreviation
#    (e.g., uppercase acronym such as NSF, NIH, NSFC).
# 2. Remove the final segment if it contains CJK characters
#    (e.g., Chinese/Japanese/Korean alias names).
# 3. Remove the final segment if it is short and looks like an alternate label.
# 4. Keep the final segment if it looks like a normal place/institution component
#    such as "Berkeley".

def is_cjk(text):
    # Check whether the text contains any CJK character
    return bool(re.search(r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]', text))

def is_abbreviation_like(text):
    cleaned = text.replace(".", "").strip()
    
    # Extract letters only
    letters = re.findall(r"[A-Za-z]", cleaned)
    
    if not letters:
        return False
    
    upper_ratio = sum(1 for ch in letters if ch.isupper()) / len(letters)
    
    # Case 1: mostly uppercase (e.g., IISc, NSF, NIH)
    if upper_ratio >= 0.6 and len(letters) <= 15:
        return True
    
    # Case 2: all uppercase pattern
    if re.fullmatch(r"[A-Z0-9&\-]{2,15}", cleaned):
        return True
    
    return False

def is_short_alias_like(text):
    # A short trailing segment may be an alias, but we do not want to
    # remove proper names like "Berkeley".
    #
    # This function is intentionally conservative.
    
    words = text.strip().split()
    
    # Remove if it is very short and contains no lowercase-rich phrase structure
    # Example candidates: "NSFC", "UGC", "RGC", "研究資助局"
    if len(words) <= 3 and len(text.strip()) <= 20:
        # If most letters are uppercase, it is likely an abbreviation
        letters = re.findall(r"[A-Za-z]", text)
        if letters:
            upper_ratio = sum(1 for ch in letters if ch.isupper()) / len(letters)
            if upper_ratio >= 0.8:
                return True
    
    return False

def should_remove_trailing_segment(segment):
    segment = segment.strip().rstrip(",").strip()
    
    if not segment:
        return False
    
    if is_cjk(segment):
        return True
    
    if is_abbreviation_like(segment):
        return True
    
    if is_short_alias_like(segment):
        return True
    
    return False

def remove_trailing_abbreviation_iterative(text):
    if not text or not isinstance(text, str):
        return text
    
    # Clean trailing commas and whitespace
    text = text.rstrip(",").strip()
    
    # Iteratively remove trailing segments
    while "," in text:
        main_part, trailing_part = text.rsplit(",", 1)
        trailing_part = trailing_part.strip()
        
        if should_remove_trailing_segment(trailing_part):
            text = main_part.rstrip(",").strip()
        else:
            break
    
    return text

# Apply to DataFrame
df["Funding_Details_NoAbbrev"] = df["Funding_Details_Clean"].apply(
    lambda lst: [remove_trailing_abbreviation_iterative(item) for item in lst]
)

# Optional: Check the result
print(df[["Funding_Details_Clean", "Funding_Details_NoAbbrev"]].head(10))

                                                                                                                Funding_Details_Clean  \
0                                                                                                                                  []   
1                                                                                                                                  []   
2                                                                                                [Hrvatska Zaklada za Znanost, HRZZ,]   
3                                                                                                                                  []   
4                                                                                                                                  []   
5                                                                                                                                  []   
6                                        

In [51]:
import re
import pandas as pd

# Step: Remove abbreviations AND blank entries
# -------------------------------------------

def is_abbreviation(text):
    if not text or not isinstance(text, str):
        return False
    
    cleaned = text.replace(".", "").strip()
    
    # Extract letters
    letters = re.findall(r"[A-Za-z]", cleaned)
    
    # Case 1: no letters (numbers or symbols only)
    # -> DO NOT treat as abbreviation
    if not letters:
        return False
    
    # Case 2: mostly uppercase letters (e.g., NSF, NIH, IISc)
    upper_ratio = sum(1 for ch in letters if ch.isupper()) / len(letters)
    if upper_ratio >= 0.6 and len(cleaned) <= 20:
        return True
    
    # Case 3: uppercase + digits (e.g., H2020, FP7)
    if re.fullmatch(r"[A-Z]{1,6}\d{1,6}", cleaned):
        return True
    
    return False


def is_blank(text):
    # Identify blank or empty values
    return (
        text is None or
        (isinstance(text, float) and pd.isna(text)) or
        (isinstance(text, str) and text.strip() == "")
    )


# Apply filtering
df["Funding_Details_Final"] = df["Funding_Details_NoAbbrev"].apply(
    lambda lst: [
        org for org in lst 
        if not is_blank(org) and not is_abbreviation(org)
    ]
)

# Optional: check results
print(df[["Funding_Details_NoAbbrev", "Funding_Details_Final"]].head(10))

                                                                                                   Funding_Details_NoAbbrev  \
0                                                                                                                        []   
1                                                                                                                        []   
2                                                                                             [Hrvatska Zaklada za Znanost]   
3                                                                                                                        []   
4                                                                                                                        []   
5                                                                                                                        []   
6                                                                                                              

In [52]:
## Write CSV file

df.to_csv('LIS_data_cleaned.csv', index=False, encoding="utf-8-sig")
print("\nCleaned data saved as 'LIS_data_cleaned.csv'")


Cleaned data saved as 'LIS_data_cleaned.csv'


In [53]:
# Step 1: Collect all funding organization names into a single list
# ----------------------------------------------------------------
all_funding_orgs = [org for sublist in df["Funding_Details_Final"] for org in sublist]

# Convert to DataFrame for easier handling
funding_df = pd.DataFrame(all_funding_orgs, columns=["Funding_Organization"])

# Total count (including duplicates)
print("Total number of funding organization entries (with duplicates):")
print(len(funding_df))

Total number of funding organization entries (with duplicates):
86811


In [54]:
# Step 2: Count frequency of each organization and save to CSV
# ------------------------------------------------------------
freq_df = (
    funding_df["Funding_Organization"]
    .value_counts()
    .reset_index()
)

freq_df.columns = ["Funding_Organization", "Frequency"]

# Save to CSV
freq_df.to_csv("funding_organization_frequency.csv", index=False, encoding="utf-8-sig")

print("\nTop 10 most frequent funding organizations:")
print(freq_df.head(10))


Top 10 most frequent funding organizations:
                                      Funding_Organization  Frequency
0             National Natural Science Foundation of China       6580
1                              National Science Foundation       3743
2                            National Institutes of Health       1594
3                         Horizon 2020 Framework Programme       1432
4       National Office for Philosophy and Social Sciences       1027
5                                      European Commission        991
6  Fundamental Research Funds for the Central Universities        943
7   National Key Research and Development Program of China        922
8               Japan Society for the Promotion of Science        788
9                              Seventh Framework Programme        755


In [55]:
# Step 3: Count unique organizations
# ---------------------------------
unique_orgs = funding_df["Funding_Organization"].unique()

print("\nTotal number of unique funding organizations:")
print(len(unique_orgs))




Total number of unique funding organizations:
21091


In [56]:
# Step 4: Randomly sample 1055 organizations (5% of the unique funding organizations)
# ------------------------------------------
# Use unique organizations for sampling (recommended)
sample_size = min(1055, len(unique_orgs))

sampled_orgs = pd.Series(unique_orgs).sample(n=sample_size, random_state=42)

# Convert to DataFrame and save
sampled_df = pd.DataFrame(sampled_orgs, columns=["Funding_Organization"])

sampled_df.to_csv("funding_organization_sample_1055.csv", index=False, encoding="utf-8-sig")

print("\nRandom sample (first 10 rows):")
print(sampled_df.head(10))


Random sample (first 10 rows):
                                                   Funding_Organization
7270                                        Lake Macquarie City Council
4788   Department of Community Medicine, Sri Devraj Urs Medical College
7442                          Swedish Foundation for Strategic Research
11233                                                     Martha Cooney
3806                                        Florida Atlantic University
11852                   Korea Research Institute of Chemical Technology
15244                                     Kompetenzzentrum Bibliometrie
7305                                          EPFL Fellows Horizon 2020
11604      National Key R&D Program of China Synthetic Biology Research
2533                    Kultúrna a Edukacná Grantová Agentúra MŠVVaŠ SR
